### Time-Lagged Cross Correlation and Window Time Lagged Cross Correlation 

The purpose of this notebook is to develop a function for computing window time lagged cross correlation on two data arrays. I will work through a synthetic example and develop the function for use on real data 

useful links: 

https://medium.com/pythoneers/cross-correlation-and-coherence-in-time-series-analysis-how-to-uncover-relationships-between-c83a08990b2d


https://www.kaggle.com/code/adepvenugopal/time-series-correlation-pearson-tlcc-dtw






In [ ]:
# import packages 
import numpy as np 
import matplotlib.pyplot as plt 
import xarray as xr 
import pandas as pd

import scipy
from scipy import signal
import colorcet as cc
import matplotlib
import cmocean 

In [ ]:
# load in function that I previously copied from Una's code to test 


## ORIGINAL FUNCTION 

def sliding_window_correlation(ds1,ds2,lag_hours,sliding_window):

    """
    - ds1: dataset used for sliding window
    - ds2: dataset used for lag hours 

    """

    # calculate when correlation analysis should start 
    start = (ds1.time[0] + pd.Timedelta(lag_hours,'h'))
    end = (ds1.time[-1] - pd.Timedelta(lag_hours,'h'))

    # calculate sliding window indice based on lag_hours 
    
    sliding_window_indice = np.arange(-lag_hours,lag_hours+1,1)
    
    # calculate sliding window start and stop 
    window_stop = start + pd.Timedelta(sliding_window,'d')

    # calculate ndays
    ndays = (end-window_stop).values.astype('timedelta64[D]').astype('int')

    # calculate correlation stuff 
    corr = []
    pval = [] 
    
    for i in np.arange(0,ndays,1):
        start = (ds1.time[0] + pd.Timedelta(lag_hours,'h')) + pd.Timedelta(i,'d')
        stop = (ds1.time[0] + pd.Timedelta(lag_hours,'h') + pd.Timedelta(sliding_window,'d')) + pd.Timedelta(i,'d') 
        
        for hrs in range(len(sliding_window_indice)):
            
            sliding_start = ds1.sel(time=start).time + pd.Timedelta(sliding_window_indice[hrs],'hr')
            sliding_stop = ds1.sel(time=stop).time + pd.Timedelta(sliding_window_indice[hrs],'hr')
            
            x = signal.detrend(ds2.sel(time=slice(start,stop)))
            y = signal.detrend(ds1.sel(time=slice(sliding_start,sliding_stop)))
        
            r, p = scipy.stats.pearsonr(x, y)
            corr.append(r)
            pval.append(p)

    # convert to good format 
    corr  = np.reshape(corr, [len(np.arange(0,ndays)),len(sliding_window_indice)]) 
    pval  = np.reshape(pval, [len(np.arange(0,ndays)),len(sliding_window_indice)]) 

    start = (ds1.time[0] + pd.Timedelta(lag_hours,'h'))
    end = (ds1.time[-1] - pd.Timedelta(lag_hours,'h'))

    start_date = (ds1.time[0] + pd.Timedelta(lag_hours,'h')).values
    date_array = [start_date + np.timedelta64(i, 'D') for i in range(ndays)]


    corr = xr.DataArray(data=corr,dims=['days','lag_hours'],
             coords=dict(days=date_array,lag_hours=sliding_window_indice))

    pval = xr.DataArray(data=pval,dims=['days','lag_hours'],
                 coords=dict(days=date_array,lag_hours=sliding_window_indice))

    return corr, pval

In [ ]:
def Plot_Corr_Pval(corr,pval,title):
    
    """ Plotting function for plotting sliding window correlations"""
    
    corr_significant = corr
    corr_significant = corr_significant.where(pval < 0.05)
    fig=plt.figure(figsize=(6,6),dpi=300) 
    
    cmap = cmocean.cm.balance  #CET_D1A #matplotlib.cm.Spectral_r
    bounds = np.arange(-0.9,1,.2)
    norm =  matplotlib.colors.BoundaryNorm(bounds, cmap.N)
    
    
    plt.pcolormesh(corr_significant.lag_hours,corr_significant.days,corr_significant,cmap = cmap, norm=norm )
    
    plt.gca().invert_yaxis()
    cbar = plt.colorbar()
    cbar.set_label('$R$')
    
    
    plt.contour(corr_significant.lag_hours,corr_significant.days,corr_significant, colors = 'k', levels = [0.6,1], linewidths = 0.3)#cmap=colorcet.cm.coolwarm)
    plt.contour(corr_significant.lag_hours,corr_significant.days,corr_significant, colors = 'k', levels = [-1,-0.6], linewidths = 0.3)#cmap=colorcet.cm.coolwarm)
    
    # plt.xlim(-100,100)
    plt.axvline(x=0,color='k',linestyle='dashed')
    
    plt.title(title)
    plt.tight_layout()



In [ ]:
# create some fake data and a fake situation 
# consider: 

### Play around with fake signals 

In [ ]:
# create a fake signal 
t = np.arange(0,100,1)
test_signal = np.sin(t)

In [ ]:
plt.plot(t,test_signal)

In [ ]:
import numpy as np
import xarray as xr

# --- Define time dimension ---
time = np.arange(np.datetime64("2020-01-01"), np.datetime64("2020-01-14"), np.timedelta64(10, "m"))

# --- Create a synthetic signal ---
# Temperature:
# Base temperature (in °C)
base_temp = 5  

# Add diurnal cycle + a slower trend + random noise
temp = (
    base_temp
    + 0.5 * np.sin(2 * np.pi * (np.arange(len(time)) / (24 * 6)))   # daily cycle
    + 0.2 * np.sin(2 * np.pi * (np.arange(len(time)) / (24 * 6 * 10)))  # slower oscillation
    + 0.05 * np.random.randn(len(time))  # random noise
)


# --- Create a lagged version (e.g., 3-hour lag) ---
lag_hours = 12
lag_samples = int(lag_hours * 60 / 10)  # because Δt = 10 minutes

da_temp_lagged = da_temp.roll(time=lag_samples, roll_coords=False)
da_temp_lagged = da_temp_lagged + 0.02 * np.random.randn(len(time))  # add independent noise

da_temp_lagged.name = "temperature_lagged"

# --- Create xarray DataArray ---
da_temp = xr.DataArray(
    temp,
    coords={"time": time},
    dims="time",
    name="temperature",
    attrs={"units": "°C", "description": "synthetic 1D temperature time series"}
)

# --- Wrap in Dataset (optional) ---
ds = xr.Dataset({"temperature": da_temp, "temperature_lagged": da_temp_lagged})

In [ ]:
fig = plt.figure(figsize=(12,4))
ds.temperature.plot()
ds.temperature_lagged.plot()

In [ ]:
temp_lagged = (
    base_temp
    + 0.5 * np.sin(2 * np.pi * (np.arange(len(time)) / (24 * 6)))   # daily cycle
    + 0.2 * np.sin(2 * np.pi * (np.arange(len(time)) / (24 * 6 * 10)))  # slower oscillation
    + 0.05 * np.random.randn(len(time))    # random noise
    
)

### Load in Some Real Data 

In [ ]:
# load in TNBD data 
tnbd_vel1000 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/TNB2018_Raw/TNBD/vel_1000.nc')
tnbd_vel1100 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/TNB2018_Raw/TNBD/vel_1100.nc')

tnbd_sbe1000 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/TNB2018_Raw/TNBD/sbe_1000.nc')
tnbd_sbe1100 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/TNB2018_Raw/TNBD/sbe_1100.nc')

In [ ]:
# load in DITD data 
ditd_vel1100 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/DITD_2018_2019/vel_1100.nc')
ditd_vel1200 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/DITD_2018_2019/vel_1200.nc')

ditd_sbe1100 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/DITD_2018_2019/sbe_1100.nc')
ditd_sbe1200 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/DITD_2018_2019/sbe_1200.nc')

In [ ]:
ditd_vel1100.east.plot()

In [ ]:
ditd_vel1200.east.plot()

In [ ]:
# FIX TIME - SUBTRACT 12 hours from upper sensor 
ditd_vel1100['time'] = ditd_vel1100.time - pd.Timedelta(12,'h')

In [ ]:
ditd_vel1100.time

In [ ]:
ditd_vel1200.time

In [ ]:
# align times 
ditd_vel1200 = ditd_vel1200.sel(time=slice('2018-03-11','2018-06-18'))
ditd_vel1100 = ditd_vel1100.sel(time=slice('2018-03-11','2018-06-18'))

In [ ]:
depth = np.array([1100,1200])

In [ ]:
# merge datasets together under a new dimension 
vel_ditd = xr.concat([ditd_vel1100,ditd_vel1200],depth).rename_dims({'concat_dim':'depth'})

In [ ]:
# plot the full time series 
fig = plt.figure(figsize=(12,4))
# plt.plot(vel.sel(time=slice('2018-04-21','2018-04-30')).time,vel.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=0).speed,label='1000')
# plt.plot(vel.sel(time=slice('2018-04-21','2018-04-30')).time,vel.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=1).speed,label='1100')
# plt.legend()

plt.plot(vel_ditd.time,vel_ditd.isel(depth=0).speed,label='1100')
plt.plot(vel_ditd.time,vel_ditd.isel(depth=1).speed,label='1200')
plt.legend()

In [ ]:
tnbd_sbe1100.temp.plot()

In [ ]:
# FIX TIME 
tnbd_vel1100['time'] = tnbd_vel1100.time - pd.Timedelta(24,'h')

In [ ]:
# resample to same time and dates
tnbd_vel1100 = tnbd_vel1100.resample(time='30min').mean()

In [ ]:
tnbd_vel1100 = tnbd_vel1100.sel(time=slice('2018-03-13','2019-01-03'))

In [ ]:
depth = np.array([1000,1100])

In [ ]:
# merge datasets together under a new dimension 
vel = xr.concat([tnbd_vel1000,tnbd_vel1100],depth).rename_dims({'concat_dim':'depth'})

In [ ]:
# plot the full time series 
fig = plt.figure(figsize=(12,4))
# plt.plot(vel.sel(time=slice('2018-04-21','2018-04-30')).time,vel.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=0).speed,label='1000')
# plt.plot(vel.sel(time=slice('2018-04-21','2018-04-30')).time,vel.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=1).speed,label='1100')
# plt.legend()

plt.plot(vel.time,vel.isel(depth=0).speed,label='1000')
plt.plot(vel.time,vel.isel(depth=1).speed,label='1100')
plt.legend()

In [ ]:
# plot a small subset of speed at each depth 
fig = plt.figure(figsize=(12,4))
plt.plot(vel.sel(time=slice('2018-04-01','2018-04-30')).time,vel.sel(time=slice('2018-04-01','2018-04-30')).isel(depth=0).speed,label='1000')
plt.plot(vel.sel(time=slice('2018-04-01','2018-04-30')).time,vel.sel(time=slice('2018-04-01','2018-04-30')).isel(depth=1).speed,label='1100')
plt.legend()

### Sliding Window Correlation - TNBD Speed 1100 and 1000 meters

In [ ]:
# attempt correlation for full time series of velocity data 
corr, pval = sliding_window_correlation(ds1=vel.isel(depth=1).speed,
                                        ds2 = vel.isel(depth=0).speed,
                                        lag_hours=72,
                                        sliding_window=14)

In [ ]:
Plot_Corr_Pval(title='TNBD Speed 1093 m and \n TNBD Speed 999 m',
               corr = corr,
               pval = pval)

# plt.xlim(20,30)

In [ ]:
# make some example plots showing in-situ velocity and aligned velocity 
fig = plt.figure(figsize=(12,6),constrained_layout=True)
plt.subplot(2,1,1)
plt.plot(vel.sel(time=slice('2018-04-21','2018-04-30')).time,vel.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=0).speed,label='1000')
plt.plot(vel.sel(time=slice('2018-04-21','2018-04-30')).time,vel.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=1).speed,label='1100')
plt.legend()
plt.title('In-Situ Data')

# plt.subplot(2,1,2)
# plt.plot(vel.sel(time=slice('2018-04-21','2018-04-30')).time,vel.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=0).speed,label='1000')
# plt.plot(vel.sel(time=slice('2018-04-21','2018-04-30')).time - pd.Timedelta(24,'h'),vel.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=1).speed,label='1100')
# plt.legend()
# plt.title('Aligned Time Series')



### Sliding Window Correlation - TNBD Zonal Velocity 1100 and 1000 meters 

In [ ]:
# attempt correlation for full time series of velocity data 
corr, pval = sliding_window_correlation(ds1=vel.isel(depth=1).east,
                                        ds2 = vel.isel(depth=0).east,
                                        lag_hours=72,
                                        sliding_window=14)

In [ ]:
Plot_Corr_Pval(title='TNBD Zonal Velocity 1093 m and \n TNBD Speed 999 m',
               corr = corr,
               pval = pval)

# plt.xlim(20,30)

In [ ]:
# make some example plots showing in-situ velocity and aligned velocity 
fig = plt.figure(figsize=(12,6),constrained_layout=True)
plt.subplot(2,1,1)
plt.plot(vel.sel(time=slice('2018-04-21','2018-04-30')).time,vel.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=0).east,label='1000')
plt.plot(vel.sel(time=slice('2018-04-21','2018-04-30')).time,vel.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=1).east,label='1100')
plt.legend()
plt.title('In-Situ Data')

plt.subplot(2,1,2)
plt.plot(vel.sel(time=slice('2018-04-21','2018-04-30')).time,vel.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=0).east,label='1000')
plt.plot(vel.sel(time=slice('2018-04-21','2018-04-30')).time - pd.Timedelta(24,'h'),vel.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=1).east,label='1100')
plt.legend()
plt.title('Aligned Time Series')



### Sliding Window Correlation - TNBD Meridional Velocity 1100 and 1000 meters

In [ ]:
# attempt correlation for full time series of velocity data 
corr, pval = sliding_window_correlation(ds1=vel.isel(depth=1).north,
                                        ds2 = vel.isel(depth=0).north,
                                        lag_hours=72,
                                        sliding_window=14)

In [ ]:
Plot_Corr_Pval(title='TNBD Meridional Velocity 1093 m and \n TNBD Speed 999 m',
               corr = corr,
               pval = pval)

# plt.xlim(20,30)

In [ ]:
# make some example plots showing in-situ velocity and aligned velocity 
fig = plt.figure(figsize=(12,6),constrained_layout=True)
plt.subplot(2,1,1)
plt.plot(vel.sel(time=slice('2018-04-21','2018-04-30')).time,vel.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=0).north,label='1000')
plt.plot(vel.sel(time=slice('2018-04-21','2018-04-30')).time,vel.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=1).north,label='1100')
plt.legend()
plt.title('In-Situ Data')

plt.subplot(2,1,2)
plt.plot(vel.sel(time=slice('2018-04-21','2018-04-30')).time,vel.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=0).north,label='1000')
plt.plot(vel.sel(time=slice('2018-04-21','2018-04-30')).time - pd.Timedelta(24,'h'),vel.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=1).north,label='1100')
plt.legend()
plt.title('Aligned Time Series')



### Sliding Window Correlation - DITD Speed 1200 and 1100 meters

In [ ]:
# attempt correlation for full time series of velocity data 
corr, pval = sliding_window_correlation(ds1=vel_ditd.isel(depth=1).speed,
                                        ds2 = vel_ditd.isel(depth=0).speed,
                                        lag_hours=72,
                                        sliding_window=14)

In [ ]:
Plot_Corr_Pval(title='DITD Speed 1200 m and \n DITD Speed 1100 m',
               corr = corr,
               pval = pval)

plt.xlim(-20,0)

In [ ]:
# make some example plots showing in-situ velocity and aligned velocity 
fig = plt.figure(figsize=(12,6),constrained_layout=True)
plt.subplot(2,1,1)
plt.plot(vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).time,vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=0).speed,label='1100')
plt.plot(vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).time,vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=1).speed,label='1200')
plt.legend()
plt.title('In-Situ Data')

plt.subplot(2,1,2)
plt.plot(vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).time,vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=0).speed,label='1100')
plt.plot(vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).time + pd.Timedelta(12.5,'h'),vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=1).speed,label='1200')
plt.legend()
plt.title('Aligned Time Series')



### Sliding Window Correlation - DITD Zonal Velocity 1200 and 1100 meters 

In [ ]:
# attempt correlation for full time series of velocity data 
corr, pval = sliding_window_correlation(ds1=vel_ditd.isel(depth=1).east,
                                        ds2 = vel_ditd.isel(depth=0).east,
                                        lag_hours=72,
                                        sliding_window=14)

In [ ]:
Plot_Corr_Pval(title='DTID Zonal Velocity 1200 m and \n TNBD Speed 1100 m',
               corr = corr,
               pval = pval)

# plt.xlim(20,30)

In [ ]:
# make some example plots showing in-situ velocity and aligned velocity 
fig = plt.figure(figsize=(12,6),constrained_layout=True)
plt.subplot(2,1,1)
plt.plot(vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).time,vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=0).east,label='1000')
plt.plot(vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).time,vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=1).east,label='1100')
plt.legend()
plt.title('In-Situ Data')

plt.subplot(2,1,2)
plt.plot(vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).time,vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=0).east,label='1000')
plt.plot(vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).time - pd.Timedelta(24,'h'),vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=1).east,label='1100')
plt.legend()
plt.title('Aligned Time Series')



### Sliding Window Correlation - DITD Meridional Velocity 1200 and 1100 meters

In [ ]:
# attempt correlation for full time series of velocity data 
corr, pval = sliding_window_correlation(ds1=vel_ditd.isel(depth=1).north,
                                        ds2 = vel_ditd.isel(depth=0).north,
                                        lag_hours=72,
                                        sliding_window=14)

In [ ]:
Plot_Corr_Pval(title='DITD Meridional Velocity 1200 m and \n TNBD Speed 1100 m',
               corr = corr,
               pval = pval)

# plt.xlim(20,30)

In [ ]:
# make some example plots showing in-situ velocity and aligned velocity 
fig = plt.figure(figsize=(12,6),constrained_layout=True)
plt.subplot(2,1,1)
plt.plot(vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).time,vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=0).east,label='1000')
plt.plot(vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).time,vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=1).east,label='1100')
plt.legend()
plt.title('In-Situ Data')

plt.subplot(2,1,2)
plt.plot(vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).time,vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=0).east,label='1000')
plt.plot(vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).time - pd.Timedelta(24,'h'),vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=1).east,label='1100')
plt.legend()
plt.title('Aligned Time Series')



### Sliding Window Correlation - DITD Speed 1100 m and TNBD Speed 1100 m 

In [ ]:
ditd_vel1100 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/DITD_2018_2019/vel_1100.nc')

In [ ]:
tnbd_vel1100 = xr.open_dataset('/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/TNB2018_Raw/TNBD/vel_1100.nc')

In [ ]:
# FIX TIME 
# FIX TIME - SUBTRACT 12 hours from upper sensor 
ditd_vel1100['time'] = ditd_vel1100.time - pd.Timedelta(12,'h')
# FIX TIME 
tnbd_vel1100['time'] = tnbd_vel1100.time - pd.Timedelta(24,'h')


In [ ]:
tnbd_vel1100 = tnbd_vel1100.sel(time=slice('2018-03-12','2018-12-31'))

In [ ]:
tnbd_vel1100

In [ ]:
ditd_vel1100 = ditd_vel1100.sel(time=slice('2018-03-12','2018-12-31')).resample(time='15min').mean()

In [ ]:
ditd_vel1100.time

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(tnbd_vel1100.time,tnbd_vel1100.speed,label='TNBD 1100 m')
plt.plot(ditd_vel1100.time,ditd_vel1100.speed, label= 'DITD 1100 m')
plt.legend()

In [ ]:
tnbd_vel1000.time

In [ ]:
ditd_vel1100 = ditd_vel1100.resample(time='30min').mean()

In [ ]:
# attempt correlation for full time series of velocity data 
corr, pval = sliding_window_correlation(ds1=ditd_vel1100.north,
                                        ds2 = tnbd_vel1000.north,
                                        lag_hours=240,
                                        sliding_window=30)

In [ ]:
Plot_Corr_Pval(title='DITD Meridional Velocity 1100 m and \n TNBD Meridional Velocity 1000 m',
               corr = corr,
               pval = pval)

# plt.xlim(20,30)
# plt.savefig('DITD_TNBD_meridional_1100_correlation.png')

In [ ]:
# make some example plots showing in-situ velocity and aligned velocity 
fig = plt.figure(figsize=(12,6),constrained_layout=True)
plt.subplot(2,1,1)
plt.plot(vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).time,vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=0).east,label='1000')
plt.plot(vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).time,vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=1).east,label='1100')
plt.legend()
plt.title('In-Situ Data')

plt.subplot(2,1,2)
plt.plot(vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).time,vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=0).east,label='1000')
plt.plot(vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).time - pd.Timedelta(24,'h'),vel_ditd.sel(time=slice('2018-04-21','2018-04-30')).isel(depth=1).east,label='1100')
plt.legend()
plt.title('Aligned Time Series')



### Calculate Shear at TNBD 

In [ ]:
# zonal shear 
zonal_shear = vel.isel(depth=1).east - vel.isel(depth=0).east

In [ ]:
merid_shear = vel.isel(depth=1).north - vel.isel(depth=0).north 

In [ ]:
shear = np.sqrt(zonal_shear**2 + merid_shear**2)

In [ ]:
zonal_shear

In [ ]:
# plot shear 
fig = plt.figure(figsize=(12,8),constrained_layout=True)

plt.subplot(3,1,1)
plt.plot(zonal_shear.time,zonal_shear)
plt.title('Zonal Shear')
plt.ylabel(r'Shear, $\frac{du}{dz}$ [s$^{-1}$]')

plt.subplot(3,1,2)
plt.plot(merid_shear.time,merid_shear)
plt.title('Meridional Shear')
plt.ylabel(r'Shear, $\frac{dv}{dz}$ [s$^{-1}$]')

plt.subplot(3,1,3)
plt.plot(shear.time,shear)
plt.title('Shear Magnitude')
plt.ylabel(r'Shear [s$^{-1}$]')

Look at relationship between presence of HSSW and shear. Look at PIPERS 

In [ ]:
## zoom into subsection of data (April - June)
fig = plt.figure(figsize=(12,8),constrained_layout=True)

plt.subplot(3,1,1)
plt.plot(zonal_shear.time.sel(time=slice('2018-04','2018-05')),zonal_shear.sel(time=slice('2018-04','2018-05')))
plt.title('Zonal Shear')
plt.ylabel(r'Shear, $\frac{du}{dz}$ [s$^{-1}$]')

plt.subplot(3,1,2)
plt.plot(merid_shear.time.sel(time=slice('2018-04','2018-05')),merid_shear.sel(time=slice('2018-04','2018-05')))
plt.title('Meridional Shear')
plt.ylabel(r'Shear, $\frac{dv}{dz}$ [s$^{-1}$]')

plt.subplot(3,1,3)
plt.plot(shear.time.sel(time=slice('2018-04','2018-05')),shear.sel(time=slice('2018-04','2018-05')))
plt.title('Shear Magnitude')
plt.ylabel(r'Shear [s$^{-1}$]')

In [ ]:
## zoom into subsection of data (September)
fig = plt.figure(figsize=(12,8),constrained_layout=True)

plt.subplot(3,1,1)
plt.plot(zonal_shear.time.sel(time=slice('2018-09-01','2018-09-30')),zonal_shear.sel(time=slice('2018-09-01','2018-09-30')))
plt.title('Zonal Shear')
plt.ylabel(r'Shear, $\frac{du}{dz}$ [s$^{-1}$]')

plt.subplot(3,1,2)
plt.plot(merid_shear.time.sel(time=slice('2018-09-01','2018-09-30')),merid_shear.sel(time=slice('2018-09-01','2018-09-30')))
plt.title('Meridional Shear')
plt.ylabel(r'Shear, $\frac{dv}{dz}$ [s$^{-1}$]')

plt.subplot(3,1,3)
plt.plot(shear.time.sel(time=slice('2018-09-01','2018-09-30')),shear.sel(time=slice('2018-09-01','2018-09-30')))
plt.title('Shear Magnitude')
plt.ylabel(r'Shear [s$^{-1}$]')

### Calculate Shear at DITD 

In [ ]:
# zonal shear 
zonal_shear = vel_ditd.isel(depth=1).east - vel_ditd.isel(depth=0).east

In [ ]:
merid_shear = vel_ditd.isel(depth=1).north - vel_ditd.isel(depth=0).north 

In [ ]:
shear = np.sqrt(zonal_shear**2 + merid_shear**2)

In [ ]:
zonal_shear

In [ ]:
# plot shear 
fig = plt.figure(figsize=(12,8),constrained_layout=True)

plt.subplot(3,1,1)
plt.plot(zonal_shear.time,zonal_shear)
plt.title('Zonal Shear')
plt.ylabel(r'Shear, $\frac{du}{dz}$ [s$^{-1}$]')

plt.subplot(3,1,2)
plt.plot(merid_shear.time,merid_shear)
plt.title('Meridional Shear')
plt.ylabel(r'Shear, $\frac{dv}{dz}$ [s$^{-1}$]')

plt.subplot(3,1,3)
plt.plot(shear.time,shear)
plt.title('Shear Magnitude')
plt.ylabel(r'Shear [s$^{-1}$]')

In [ ]:
## zoom into subsection of data (April - June)
fig = plt.figure(figsize=(12,8),constrained_layout=True)

plt.subplot(3,1,1)
plt.plot(zonal_shear.time.sel(time=slice('2018-04','2018-05')),zonal_shear.sel(time=slice('2018-04','2018-05')))
plt.title('Zonal Shear')
plt.ylabel(r'Shear, $\frac{du}{dz}$ [s$^{-1}$]')

plt.subplot(3,1,2)
plt.plot(merid_shear.time.sel(time=slice('2018-04','2018-05')),merid_shear.sel(time=slice('2018-04','2018-05')))
plt.title('Meridional Shear')
plt.ylabel(r'Shear, $\frac{dv}{dz}$ [s$^{-1}$]')

plt.subplot(3,1,3)
plt.plot(shear.time.sel(time=slice('2018-04','2018-05')),shear.sel(time=slice('2018-04','2018-05')))
plt.title('Shear Magnitude')
plt.ylabel(r'Shear [s$^{-1}$]')

In [ ]:
## zoom into subsection of data (September)
fig = plt.figure(figsize=(12,8),constrained_layout=True)

plt.subplot(3,1,1)
plt.plot(zonal_shear.time.sel(time=slice('2018-04-22','2018-05-22')),zonal_shear.sel(time=slice('2018-04-22','2018-05-22')))
plt.title('Zonal Shear')
plt.ylabel(r'Shear, $\frac{du}{dz}$ [s$^{-1}$]')

plt.subplot(3,1,2)
plt.plot(merid_shear.time.sel(time=slice('2018-04-22','2018-05-22')),merid_shear.sel(time=slice('2018-04-22','2018-05-22')))
plt.title('Meridional Shear')
plt.ylabel(r'Shear, $\frac{dv}{dz}$ [s$^{-1}$]')

plt.subplot(3,1,3)
plt.plot(shear.time.sel(time=slice('2018-04-22','2018-05-22')),shear.sel(time=slice('2018-04-22','2018-05-22')))
plt.title('Shear Magnitude')
plt.ylabel(r'Shear [s$^{-1}$]')

### Look at Correlation Between Wind Speed and Salinity - Reproduce Una's Plot 

In [ ]:
import os

In [ ]:
# load data 
directory = '/Users/mackenzie/Documents/Research/PhD/TNB_Research/data/netCDFs'
varnames=[]
for filename in os.listdir(directory):
    if filename.endswith(".nc"):
        file=os.path.splitext(os.path.basename(filename))[0]
        varnames.append(file)

for name in varnames:
    globals()[name] = xr.open_dataset(directory+'/'+name+'.nc')
    print(name);

In [ ]:
# load data 
sbe = ldeo_mooring_tempsal_updated


In [ ]:
aws_data = aws_data.assign(wspd = aws_data.wspd.dropna(dim='time'))

In [ ]:
aws_data.wspd

In [ ]:
sal = sbe.isel(depth=0).practical_salinity

In [ ]:
wspd = aws_data.wspd

In [ ]:
sal = sal.resample(time='10min').mean().sel(time=slice('2017-02-02','2018-03-04'))

In [ ]:
# select same time in wspd data 
wspd = wspd.sel(time=slice('2017-02-02','2018-03-04'))

In [ ]:
plt.plot(wspd)

In [ ]:
plt.plot(sal)

In [ ]:
wspd.time

In [ ]:
sal.time

In [ ]:
wspd = wspd.dropna(dim='time')

In [ ]:
sal = sal.sel(time=wspd.time)

In [ ]:
sal = sal.dropna(dim='time')

In [ ]:
wspd = wspd.sel(time=sal.time)

In [ ]:
time = wspd.time.data

In [ ]:
ds = xr.Dataset(
    data_vars = dict(
        wspd=(['time'],wspd.data),
        sal = (['time'],sal.data)
    ),
    coords = dict(time=time))

In [ ]:
ds

In [ ]:
ds1 = ds.wspd
ds2 = ds.sal

In [ ]:
plt.plot(ds1.time,ds2.time)

In [ ]:
ds1

In [ ]:
ds2

In [ ]:
lag_hours = 200
sliding_window = 30*24     #hours

In [ ]:
# calculate when correlation analysis should start 
start = (ds1.time[0] + pd.Timedelta(lag_hours,'h'))
end = (ds1.time[-1] - pd.Timedelta(lag_hours,'h'))

In [ ]:
start

In [ ]:
end

In [ ]:
sliding_window_indice = np.arange(-lag_hours,lag_hours+1,1)

In [ ]:
sliding_window_indice

In [ ]:
# calculate sliding window start and stop 
window_stop = start + pd.Timedelta(sliding_window,'hr')

In [ ]:
window_stop

In [ ]:
(end-window_stop).values.astype('timedelta64[D]').astype('int')

In [ ]:
nhours = (end-window_stop).values.astype('timedelta64[h]').astype('int')

In [ ]:
# # calculate sliding window start and stop 
# window_stop = start + pd.Timedelta(sliding_window,'hr')

# # calculate ndays
# ndays = (end-window_stop).values.astype('timedelta64[D]').astype('int')

# calculate correlation stuff 
corr = []
pval = []

In [ ]:
ndays

In [ ]:
window_stop

In [ ]:
# test one window to see what happens with slicing 
start = (ds1.time[0] + pd.Timedelta(lag_hours,'h')) + pd.Timedelta(1,'d')
stop = (ds1.time[0] + pd.Timedelta(lag_hours,'h') + pd.Timedelta(sliding_window,'d')) + pd.Timedelta(1,'d')

In [ ]:
start

In [ ]:
stop

In [ ]:
sliding_start = ds1.sel(time=start).time + pd.Timedelta(sliding_window_indice[0],'hr')
sliding_stop = ds1.sel(time=stop).time + pd.Timedelta(sliding_window_indice[0],'hr')

In [ ]:
sliding_start

In [ ]:
sliding_stop

In [ ]:
ds2.sel(time=slice(start,stop))

In [ ]:
ds1.sel(time=slice(sliding_start,sliding_stop))

In [ ]:
for i in np.arange(0,nhours,1):
    start = (ds1.time[0] + pd.Timedelta(lag_hours,'h')) + pd.Timedelta(i,'h')
    stop = (ds1.time[0] + pd.Timedelta(lag_hours,'h') + pd.Timedelta(sliding_window,'h')) + pd.Timedelta(i,'h') 
    
    for hrs in range(len(sliding_window_indice)):
        
        sliding_start = ds1.sel(time=start).time + pd.Timedelta(sliding_window_indice[hrs],'hr')
        sliding_stop = ds1.sel(time=stop).time + pd.Timedelta(sliding_window_indice[hrs],'hr')
        
        x = signal.detrend(ds2.sel(time=slice(start,stop)))
        y = signal.detrend(ds1.sel(time=slice(sliding_start,sliding_stop)))
    
        r, p = scipy.stats.pearsonr(x, y)
        corr.append(r)
        pval.append(p)

In [ ]:
len(x)

In [ ]:
len(y)

In [ ]:
sliding_start

In [ ]:
sliding_stop

In [ ]:
start

In [ ]:
stop

In [ ]:
len(x)

In [ ]:
len(y)

In [ ]:
print(ds1.time[:5].values)

In [ ]:
wspd.sel(time=slice('2017-05-01','2017-05-14')).plot()

In [ ]:
sal.sel(time=slice('2017-05-01','2017-05-14')).plot()

In [ ]:
sal = sal.sel(time=slice('2017-03','2017-12'))

In [ ]:
wspd = wspd.sel(time=slice('2017-03','2017-12'))

In [ ]:
sal = sal.interpolate_na(dim='time')

In [ ]:
wspd = wspd.interpolate_na(dim='time')

In [ ]:
# now we can do correlation analysis since wind speed and sal data are aligned 

corr, pval = sliding_window_correlation(ds1=wspd,
                                        ds2 = sal,
                                        lag_hours=200,
                                        sliding_window=30)

In [ ]:
Plot_Corr_Pval(corr,pval,title='Wind Speed and 47 m Salinity')
plt.xlim(-50,0)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12,4))

ax1.plot(wspd.time.sel(time=slice('2017-05-01','2017-05-21')) + pd.Timedelta(15,'h'),wspd.sel(time=slice('2017-05-01','2017-05-21')),color='orange')
# ax1.plot(shear.time,shear.isel(distance=slice(0,5)).mean('distance').rolling(time=720,center=True).mean(),color='blue')
# ax1.plot(shear.time,shear.isel(distance=slice(5,11)).mean('distance').rolling(time=720,center=True).mean())
ax1.set_ylabel(r' Wind Speed m/s', color='orange')
ax1.tick_params(axis='y', labelcolor='orange')

ax2 = ax1.twinx()
ax2.plot(sal.time.sel(time=slice('2017-05-01','2017-05-21')),sal.sel(time=slice('2017-05-01','2017-05-21')),color='blue')
ax2.set_ylabel('Sal',color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

ax1.tick_params(axis='x',labelsize=10,labelrotation=30)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12,4))

ax1.plot(wspd.time.sel(time=slice('2017-07-24','2017-08-23')) + pd.Timedelta(30,'h'),wspd.sel(time=slice('2017-07-24','2017-08-23')),color='orange')
# ax1.plot(shear.time,shear.isel(distance=slice(0,5)).mean('distance').rolling(time=720,center=True).mean(),color='blue')
# ax1.plot(shear.time,shear.isel(distance=slice(5,11)).mean('distance').rolling(time=720,center=True).mean())
ax1.set_ylabel(r' Wind Speed m/s', color='orange')
ax1.tick_params(axis='y', labelcolor='orange')

ax2 = ax1.twinx()
ax2.plot(sal.time.sel(time=slice('2017-07-24','2017-08-23')),sal.sel(time=slice('2017-07-24','2017-08-23')),color='blue')
ax2.set_ylabel('Sal',color='blue')
ax2.tick_params(axis='y', labelcolor='blue')

ax1.tick_params(axis='x',labelsize=10,labelrotation=30)
ax2.set_ylim(34.7,34.85)

The problem I was having with the sliding window function was due to dropping nans in the data arrays. I think this messed up the time axis. The better way to handle the nans is to linearly interpolate them so the time dimension does not get messed up and cause issues when slicing. 

### Other stuff 

In [ ]:
# the above is a nice subset where we can clearly see that the 1100 m velocity lags the 1000 m velocity by about one day
# define this subset below to use for correlation testing example 
subset = vel.sel(time=slice('2018-04-13','2018-04-30'))

In [ ]:
corr, pval = sliding_window_correlation(ds1=subset.isel(depth=1).speed,ds2=subset.isel(depth=0).speed,lag_hours=48,sliding_window=4)

In [ ]:
corr_significant = corr
corr_significant = corr_significant.where(pval < 0.05)
fig=plt.figure(figsize=(6,6),dpi=300) 

cmap = cc.cm.CET_D1_r#CET_D1A #matplotlib.cm.Spectral_r
bounds = np.arange(-0.9,1,.2)
norm =  matplotlib.colors.BoundaryNorm(bounds, cmap.N)


plt.pcolormesh(corr_significant.lag_hours,corr_significant.days,corr_significant,cmap = cmap, norm=norm)

plt.gca().invert_yaxis()
cbar = plt.colorbar()
cbar.set_label('$R$')


plt.contour(corr_significant.lag_hours,corr_significant.days,corr_significant, colors = 'k', levels = [0.6,1], linewidths = 0.3)#cmap=colorcet.cm.coolwarm)
plt.contour(corr_significant.lag_hours,corr_significant.days,corr_significant, colors = 'k', levels = [-1,-0.6], linewidths = 0.3)#cmap=colorcet.cm.coolwarm)

# plt.xlim(-100,100)
plt.axvline(x=0,color='k',linestyle='dashed')

plt.title('TNBD Speed 1093 m and \n TNBD Speed 999 m')
plt.tight_layout()
plt.xlim(-40,40)

In [ ]:
np.correlate(vel.isel(depth=0).speed,vel.isel(depth=1).speed)

In [ ]:
# 1. Calculate cross-correlation using scipy.signal.correlate
# 'full' mode returns the convolution at each point of overlap
from scipy import correlate 
correlation_output = correlate(subset.isel(depth=0).speed - np.mean(subset.isel(depth=0).speed), subset.isel(depth=1).speed - np.mean(subset.isel(depth=1).speed), mode='full')

# 2. Determine the lags corresponding to the correlation output
# The lags range from -(len(series2)-1) to (len(series1)-1)
lags = np.arange(-(len(subset.isel(depth=0).speed) - 1), len(subset.isel(depth=1).speed))

# 3. Find the lag with the maximum absolute correlation
max_corr_index = np.argmax(np.abs(correlation_output))
optimal_lag = lags[max_corr_index]
max_correlation_value = correlation_output[max_corr_index]

In [ ]:
optimal_lag